# EfficientNetB0 — Fair Comparison Run

**Model:** Chris Deotte's EfficientNetB0 starter (LB 0.43), faithful to the original recipe.
Source: https://www.kaggle.com/code/cdeotte/efficientnetb0-starter-lb-0-43

**Purpose:** Produce per-fold OOF predictions in the shared fair-comparison format so this
model can be compared head-to-head with VIPEEGNet and TCS-Net.

**Fairness alignment with the other two models:**
- 5-fold GroupKFold on `patient_id` — loaded from `patient_folds.csv` (same patients per fold
  as VIPEEGNet and TCS-Net).
- KL-Divergence loss (native to Deotte's recipe; same metric all three use).
- HQ evaluation subset = eeg_ids where at least one segment has votes ≥ 10.
- Seed documented (SEED=2024). GroupKFold is deterministic.

**Fair-comparison output schema (shared with VIPEEGNet and TCS-Net):**
- Each `fold{k}_oof.npz` saves: `yt`, `yp`, `wt`, `yp_raw_logits`, `eeg_ids`,
  `spectrogram_ids`, `hq_mask`, `max_segment_votes`, `temperature=1.0`.
- `yp_raw_logits = log(yp)` — equivalent to true logits under any downstream softmax,
  which is all the comparison notebook will do. No TTA or temperature scaling is applied
  (native Deotte recipe), so `yp_native == yp_raw`.

**Native recipe preserved:**
- EfficientNetB0 backbone with ImageNet weights.
- 128×256×8 input = 4 Kaggle spectrograms + 4 EEG-derived mel spectrograms.
- Reshape to 512×512×3 monotone (Deotte's trick).
- Mel-spec params: `n_fft=1024, hop_length=len(x)//256, n_mels=128, fmin=0, fmax=20, win_length=128`.
- `eeg_id` aggregation (votes summed across segments of the same eeg_id).
- Adam 1e-3, step LR `1e-3 → 1e-4 → 1e-5`, 4 epochs, BS=32, mixed precision.
- No augmentation (Deotte's baseline).

**Target:** Deotte reports CV 0.59 KL-Div on this recipe.


## 1. Environment


In [1]:
import os, gc, sys, math, time, json, warnings
from pathlib import Path

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL']  = '3'
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
print(f'TensorFlow {tf.__version__} | GPUs: {tf.config.list_physical_devices("GPU")}')

# memory growth
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

# Mixed precision (Deotte's recipe uses auto_mixed_precision; explicit policy is equivalent and safer)
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f'Mixed precision: {tf.keras.mixed_precision.global_policy().name}')

# Determinism (documented; GroupKFold is the main source of split determinism)
SEED = 2024
np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

E0000 00:00:1776876812.437648  123070 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776876812.442720  123070 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776876812.457214  123070 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776876812.457226  123070 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776876812.457229  123070 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776876812.457230  123070 computation_placer.cc:177] computation placer already registered. Please check linka

TensorFlow 2.19.0 | GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Mixed precision: mixed_float16


In [2]:
# ---------- paths ----------
DATA_DIR   = '/workspace/hms-data'
EEG_DIR    = f'{DATA_DIR}/train_eegs'
SPEC_DIR   = f'{DATA_DIR}/train_spectrograms'

FAIR_DIR   = '/workspace/fair_comparison'
OUT_DIR    = f'{FAIR_DIR}/effnetb0'
WEIGHTS    = f'{OUT_DIR}/weights'

# Caches for the two spectrogram types (built once, reused)
KSPEC_CACHE = '/workspace/kaggle_spec_cache'   # Kaggle-provided 10-min spectrograms
EEGSPEC_CACHE = '/workspace/eeg_spec_cache'    # Mel spectrograms from raw 50s EEG

for d in [OUT_DIR, WEIGHTS, KSPEC_CACHE, EEGSPEC_CACHE]:
    Path(d).mkdir(parents=True, exist_ok=True)

# ---------- hyperparameters (Deotte's native recipe) ----------
N_FOLDS       = 5
BATCH_SIZE    = 32
BATCH_VAL     = 64
EPOCHS        = 4
LR_MAX        = 1e-3
LR_STEP_DECAY = 0.1
LR_SUSTAIN    = 1    # first 1 epoch at LR_MAX, then step decay

USE_KAGGLE_SPECTROGRAMS = True
USE_EEG_SPECTROGRAMS    = True

TARGETS     = ['seizure_vote','lpd_vote','gpd_vote','lrda_vote','grda_vote','other_vote']
CLASS_NAMES = ['Seizure','LPD','GPD','LRDA','GRDA','Other']

# Path sanity
assert os.path.isfile(f'{DATA_DIR}/train.csv'), f'train.csv not at {DATA_DIR}'
assert os.path.isdir(EEG_DIR),  f'train_eegs/ not at {EEG_DIR}'
assert os.path.isdir(SPEC_DIR), f'train_spectrograms/ not at {SPEC_DIR}'
assert os.path.isfile(f'{FAIR_DIR}/patient_folds.csv'), \
    'Run 00_generate_fair_splits.ipynb first to create patient_folds.csv'

print('Paths OK.')
print(f'  OUT_DIR      = {OUT_DIR}')
print(f'  KSPEC_CACHE  = {KSPEC_CACHE}')
print(f'  EEGSPEC_CACHE= {EEGSPEC_CACHE}')

Paths OK.
  OUT_DIR      = /workspace/fair_comparison/effnetb0
  KSPEC_CACHE  = /workspace/kaggle_spec_cache
  EEGSPEC_CACHE= /workspace/eeg_spec_cache


## 2. Data — Deotte's eeg_id aggregation, merged with shared patient folds


In [3]:
df_full = pd.read_csv(f'{DATA_DIR}/train.csv')
print(f'Full manifest: {len(df_full):,} segments | {df_full["eeg_id"].nunique():,} eeg_ids | '
      f'{df_full["patient_id"].nunique():,} patients')

# Deotte: one row per eeg_id, with min/max spectrogram offsets, votes summed, patient_id
train = df_full.groupby('eeg_id')[['spectrogram_id','spectrogram_label_offset_seconds']].agg(
    {'spectrogram_id':'first','spectrogram_label_offset_seconds':'min'})
train.columns = ['spec_id','min']
train['max'] = df_full.groupby('eeg_id')['spectrogram_label_offset_seconds'].max()
train['patient_id'] = df_full.groupby('eeg_id')['patient_id'].first()

# Vote aggregation: sum across all segments of the same eeg_id
tmp = df_full.groupby('eeg_id')[TARGETS].agg('sum')
for t in TARGETS:
    train[t] = tmp[t].values

# For HQ evaluation — the MAX segment-level vote count per eeg_id. If any segment had
# >=10 experts, we call this eeg_id HQ.
df_full['total_votes_segment'] = df_full[TARGETS].sum(1)
train['max_segment_votes'] = df_full.groupby('eeg_id')['total_votes_segment'].max()

# Raw vote sum (for sample weight, following Deotte's spirit — he doesn't weight, we keep no weight)
train['total_votes_sum'] = train[TARGETS].sum(1)

# Normalize to probability distribution (Deotte's exact step)
y_data = train[TARGETS].values.astype(np.float32)
y_data = y_data / y_data.sum(axis=1, keepdims=True)
train[TARGETS] = y_data

train['target_class'] = df_full.groupby('eeg_id')['expert_consensus'].first()
train = train.reset_index()

print(f'\nAggregated: {len(train):,} unique eeg_ids')
print(f'HQ eeg_ids (at least one segment votes>=10): '
      f'{(train["max_segment_votes"] >= 10).sum():,}')

Full manifest: 106,800 segments | 17,089 eeg_ids | 1,950 patients

Aggregated: 17,089 unique eeg_ids
HQ eeg_ids (at least one segment votes>=10): 5,939


In [4]:
# Merge shared patient folds
pf = pd.read_csv(f'{FAIR_DIR}/patient_folds.csv')
train = train.merge(pf, on='patient_id', how='left')
assert train['fold'].notna().all(), 'some eeg_ids have no fold — patient_folds.csv is incomplete'
train['fold'] = train['fold'].astype(int)

print(f'{"Fold":>4} | {"eeg_ids":>8} | {"HQ":>7} | {"patients":>8}')
print('-' * 40)
for k in range(N_FOLDS):
    fd = train[train.fold == k]
    hq = fd[fd.max_segment_votes >= 10]
    print(f'{k:>4} | {len(fd):>8,} | {len(hq):>7,} | {fd["patient_id"].nunique():>8,}')

train.head()

Fold |  eeg_ids |      HQ | patients
----------------------------------------
   0 |    3,877 |   1,296 |      390
   1 |    3,100 |   1,219 |      390
   2 |    3,517 |   1,144 |      390
   3 |    3,438 |   1,189 |      390
   4 |    3,157 |   1,091 |      390


,eeg_id,spec_id,min,max,patient_id,seizure_vote,lpd_vote,gpd_vote,lrda_vote,grda_vote,other_vote,max_segment_votes,total_votes_sum,target_class,fold
0,568657,789577333,0.0,16.0,20654,0.0,0.000000,0.25,0.000000,0.166667,0.583333,12,48,Other,0
1,582999,1552638400,0.0,38.0,20230,0.0,0.857143,0.00,0.071429,0.000000,0.071429,14,154,LPD,0
2,642382,14960202,1008.0,1032.0,5955,0.0,0.000000,0.00,0.000000,0.000000,1.000000,1,2,Other,2
3,751790,618728447,908.0,908.0,38549,0.0,0.000000,1.00,0.000000,0.000000,0.000000,1,1,GPD,3
4,778705,52296320,0.0,0.0,40955,0.0,0.000000,0.00,0.000000,0.000000,1.000000,2,2,Other,4


## 3. Build the spectrogram caches

**Two caches needed:**

1. **Kaggle spectrogram cache**: one `.npy` per `spectrogram_id`, shape `(T, 400)`, parsed
   from the Kaggle parquet file (all time points, all 400 frequency columns, stored once).
2. **EEG spectrogram cache**: one `.npy` per `eeg_id`, shape `(128, 256, 4)`, Deotte's exact
   mel-spec-from-EEG construction (middle 50 seconds, 4 brain regions LL/LP/RP/RR).

Both caches are reused across folds, so this is a one-time ~30-60 min cost.


In [5]:
# Cache #1 — Kaggle spectrograms (one .npy per spec_id, ALL columns)
from tqdm.auto import tqdm

spec_ids = train['spec_id'].unique()
cached = sum(1 for s in spec_ids if os.path.exists(f'{KSPEC_CACHE}/{s}.npy'))
print(f'Kaggle spectrograms: need {len(spec_ids):,}, cached {cached:,}')

if cached < len(spec_ids):
    print('Caching Kaggle spectrograms...')
    for sid in tqdm(spec_ids, desc='Kaggle spec'):
        out = f'{KSPEC_CACHE}/{sid}.npy'
        if os.path.exists(out):
            continue
        try:
            tmp = pd.read_parquet(f'{SPEC_DIR}/{sid}.parquet')
            # drop the time column if present, keep the 400 freq columns
            arr = tmp.iloc[:, 1:].values.astype(np.float32)
            np.save(out, arr)
        except Exception as e:
            print(f'  warn: {sid} failed: {e}')
            np.save(out, np.zeros((600, 400), dtype=np.float32))
    print('Done.')
else:
    print('All Kaggle spectrograms cached.')

Kaggle spectrograms: need 11,138, cached 11,138
All Kaggle spectrograms cached.


In [6]:
import sys
!{sys.executable} -m pip install pywavelets librosa -q

In [7]:
# Cache #2 — EEG-derived mel spectrograms (Deotte's exact function)
import pywt  # not strictly required unless USE_WAVELET is set
import librosa

USE_WAVELET = None  # Deotte's default — no wavelet denoise in the baseline

NAMES = ['LL', 'LP', 'RP', 'RR']
FEATS = [
    ['Fp1','F7','T3','T5','O1'],   # LL
    ['Fp1','F3','C3','P3','O1'],   # LP
    ['Fp2','F8','T4','T6','O2'],   # RP
    ['Fp2','F4','C4','P4','O2'],   # RR
]

def maddest(d, axis=None):
    return np.mean(np.absolute(d - np.mean(d, axis)), axis)

def denoise(x, wavelet='haar', level=1):
    coeff = pywt.wavedec(x, wavelet, mode='per')
    sigma = (1/0.6745) * maddest(coeff[-level])
    uthresh = sigma * np.sqrt(2*np.log(len(x)))
    coeff[1:] = (pywt.threshold(i, value=uthresh, mode='hard') for i in coeff[1:])
    return pywt.waverec(coeff, wavelet, mode='per')

def spectrogram_from_eeg(parquet_path):
    '''Deotte's spectrogram_from_eeg, verbatim (display branch removed).'''
    eeg = pd.read_parquet(parquet_path)
    middle = (len(eeg) - 10_000) // 2
    eeg = eeg.iloc[middle:middle + 10_000]

    img = np.zeros((128, 256, 4), dtype='float32')
    for k in range(4):
        COLS = FEATS[k]
        for kk in range(4):
            x = eeg[COLS[kk]].values - eeg[COLS[kk+1]].values
            m = np.nanmean(x)
            if np.isnan(x).mean() < 1:
                x = np.nan_to_num(x, nan=m)
            else:
                x[:] = 0
            if USE_WAVELET:
                x = denoise(x, wavelet=USE_WAVELET)
            mel_spec = librosa.feature.melspectrogram(
                y=x, sr=200, hop_length=len(x)//256,
                n_fft=1024, n_mels=128, fmin=0, fmax=20, win_length=128,
            )
            width = (mel_spec.shape[1] // 32) * 32
            mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max).astype(np.float32)[:, :width]
            mel_spec_db = (mel_spec_db + 40) / 40
            img[:, :, k] += mel_spec_db
        img[:, :, k] /= 4.0
    return img

print('spectrogram_from_eeg defined (Deotte\'s function, verbatim).')

spectrogram_from_eeg defined (Deotte's function, verbatim).


In [8]:
# Build the EEG spec cache
eeg_ids = train['eeg_id'].unique()
cached = sum(1 for e in eeg_ids if os.path.exists(f'{EEGSPEC_CACHE}/{e}.npy'))
print(f'EEG spectrograms: need {len(eeg_ids):,}, cached {cached:,}')

if cached < len(eeg_ids):
    print('Caching EEG spectrograms (~20-40 min)...')
    for eid in tqdm(eeg_ids, desc='EEG spec'):
        out = f'{EEGSPEC_CACHE}/{eid}.npy'
        if os.path.exists(out):
            continue
        try:
            img = spectrogram_from_eeg(f'{EEG_DIR}/{eid}.parquet')
            np.save(out, img.astype(np.float32))
        except Exception as e:
            print(f'  warn: {eid} failed: {e}')
            np.save(out, np.zeros((128, 256, 4), dtype=np.float32))
    print('Done.')
else:
    print('All EEG spectrograms cached.')

EEG spectrograms: need 17,089, cached 17,089
All EEG spectrograms cached.


In [9]:
# Load both caches into RAM dicts (Deotte loads .npy pickles; we use the cache dir)
print('Loading caches into memory...')

spectrograms = {}
for sid in tqdm(spec_ids, desc='Kaggle spec load'):
    spectrograms[sid] = np.load(f'{KSPEC_CACHE}/{sid}.npy')

all_eegs = {}
for eid in tqdm(eeg_ids, desc='EEG spec load'):
    all_eegs[eid] = np.load(f'{EEGSPEC_CACHE}/{eid}.npy')

print(f'Loaded {len(spectrograms):,} Kaggle specs, {len(all_eegs):,} EEG specs.')
print(f'Example Kaggle shape: {next(iter(spectrograms.values())).shape}')
print(f'Example EEG shape:    {next(iter(all_eegs.values())).shape}')

Loading caches into memory...


Kaggle spec load:   0%|          | 0/11138 [00:00<?, ?it/s]

EEG spec load:   0%|          | 0/17089 [00:00<?, ?it/s]

Loaded 11,138 Kaggle specs, 17,089 EEG specs.
Example Kaggle shape: (308, 400)
Example EEG shape:    (128, 256, 4)


## 4. DataGenerator — Deotte's original, verbatim

No augmentation (Deotte's baseline runs without). The `__getitem__` produces `(B, 128, 256, 8)`
tensors: first 4 channels are Kaggle spectrograms, last 4 are EEG-derived.


In [10]:
class DataGenerator(tf.keras.utils.Sequence):
    '''Verbatim port of Deotte\'s DataGenerator, paths adapted.'''

    def __init__(self, data, batch_size=32, shuffle=False, augment=False, mode='train',
                 specs=None, eeg_specs=None):
        self.data       = data
        self.batch_size = batch_size
        self.shuffle    = shuffle
        self.augment    = augment
        self.mode       = mode
        self.specs      = specs if specs is not None else spectrograms
        self.eeg_specs  = eeg_specs if eeg_specs is not None else all_eegs
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        idx = self.indexes[index * self.batch_size : (index + 1) * self.batch_size]
        X, y = self.__data_generation(idx)
        return X, y

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.data))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, idx):
        X = np.zeros((len(idx), 128, 256, 8), dtype='float32')
        y = np.zeros((len(idx), 6), dtype='float32')

        for j, i in enumerate(idx):
            row = self.data.iloc[i]
            r = 0 if self.mode == 'test' else int((row['min'] + row['max']) // 4)

            # --- Kaggle spectrograms: 4 region slices, log + standardize ---
            for k in range(4):
                img = self.specs[row.spec_id][r : r + 300, k*100 : (k+1)*100].T
                img = np.clip(img, np.exp(-4), np.exp(8))
                img = np.log(img)
                ep  = 1e-6
                m   = np.nanmean(img.flatten())
                s   = np.nanstd(img.flatten())
                img = (img - m) / (s + ep)
                img = np.nan_to_num(img, nan=0.0)
                X[j, 14:-14, :, k] = img[:, 22:-22] / 2.0

            # --- EEG mel spectrograms (pre-computed, 128x256x4) ---
            X[j, :, :, 4:] = self.eeg_specs[row.eeg_id]

            if self.mode != 'test':
                y[j] = row[TARGETS]

        return X, y

print('DataGenerator defined (Deotte\'s original).')

DataGenerator defined (Deotte's original).


## 5. Model — EfficientNetB0, 8-ch → 512×512×3 reshape

Deotte's construction: stack the 4 Kaggle spec channels along height, stack the 4 EEG spec
channels along height, concatenate along width → 512×512×1 → replicate to 3 channels → B0.


In [11]:
# Install and import the efficientnet package that Deotte used
# (tf.keras.applications.EfficientNetB0 is acceptable equivalent; Deotte uses the efficientnet pkg
# because it had autoaugment weights. Either works for fair comparison — we use tf.keras for portability.)
import tensorflow.keras.applications.efficientnet as effn

def build_model():
    inp = tf.keras.Input(shape=(128, 256, 8))

    # Split into Kaggle (0..3) and EEG (4..7), stack each vertically -> (512, 256, 1)
    x1 = [inp[:, :, :, i:i+1] for i in range(4)]
    x1 = tf.keras.layers.Concatenate(axis=1)(x1)
    x2 = [inp[:, :, :, i+4:i+5] for i in range(4)]
    x2 = tf.keras.layers.Concatenate(axis=1)(x2)

    # Horizontal concat -> (512, 512, 1); replicate to 3 channels
    if USE_KAGGLE_SPECTROGRAMS and USE_EEG_SPECTROGRAMS:
        x = tf.keras.layers.Concatenate(axis=2)([x1, x2])
    elif USE_EEG_SPECTROGRAMS:
        x = x2
    else:
        x = x1
    x = tf.keras.layers.Concatenate(axis=3)([x, x, x])

    # Base model — EfficientNetB0, ImageNet weights
    base = effn.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(512, 512, 3))
    x = base(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(6, activation='softmax', dtype='float32')(x)  # float32 head for AMP safety

    model = tf.keras.Model(inp, x)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR_MAX),
        loss=tf.keras.losses.KLDivergence(),
    )
    return model

_m = build_model()
print(f'Params: {_m.count_params():,}')
del _m
gc.collect()

I0000 00:00:1776876840.134122  123070 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22282 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:e1:00.0, compute capability: 8.9


Params: 4,057,257


25915

## 6. LR schedule — Deotte's step schedule


In [12]:
LR_START = 1e-4
LR_RAMPUP  = 0
LR_SUSTAIN_EPOCHS = 1

def lrfn(epoch):
    if epoch < LR_RAMPUP:
        return (LR_MAX - LR_START) / max(LR_RAMPUP, 1) * epoch + LR_START
    if epoch < LR_RAMPUP + LR_SUSTAIN_EPOCHS:
        return LR_MAX
    return LR_MAX * (LR_STEP_DECAY ** ((epoch - LR_RAMPUP - LR_SUSTAIN_EPOCHS) // 1))

rng = list(range(EPOCHS))
print('LR per epoch:', [f'{lrfn(e):.1e}' for e in rng])
LR_CB = tf.keras.callbacks.LearningRateScheduler(lrfn, verbose=1)

LR per epoch: ['1.0e-03', '1.0e-03', '1.0e-04', '1.0e-05']


## 7. Train all 5 folds


In [13]:
from sklearn.metrics import roc_auc_score

def kld_np(p, t, eps=1e-7):
    p = np.clip(p, eps, 1.0); t = np.clip(t, eps, 1.0)
    return float(np.mean(np.sum(t * np.log(t / p), axis=1)))

# Shared-schema summary keys, consistent with VIPEEGNet and TCS-Net:
# - 'kld_hq_native' and 'kld_hq_raw' are identical here (no TTA, no temperature)
# - 'yp_raw_logits' = log(yp), equivalent to true logits for downstream softmax
# - 'spectrogram_ids' saved alongside 'eeg_ids' for cross-model join
summary = {
    'model':        'EfficientNetB0-Deotte',
    'recipe':       'Deotte LB 0.43 — 8ch spec, KLD, Adam 1e-3, step LR, 4 epochs, BS=32',
    'n_folds':      N_FOLDS,
    'batch_size':   BATCH_SIZE,
    'epochs':       EPOCHS,
    'seed':         SEED,
    'fair_splits':  'patient_folds.csv',
    'tta':          'none',
    'per_fold':     [],
}

t_start_all = time.time()
for k in range(N_FOLDS):
    print('\n' + '#' * 40)
    print(f'### Fold {k}')
    print('#' * 40)

    tr_df = train[train.fold != k].reset_index(drop=True)
    vl_df = train[train.fold == k].reset_index(drop=True)
    print(f'Train {len(tr_df):,} eeg_ids | Valid {len(vl_df):,} eeg_ids')

    tr_gen = DataGenerator(tr_df, batch_size=BATCH_SIZE, shuffle=True,  mode='train')
    vl_gen = DataGenerator(vl_df, batch_size=BATCH_VAL,  shuffle=False, mode='valid')

    tf.keras.backend.clear_session()
    gc.collect()
    model = build_model()

    t0 = time.time()
    hist = model.fit(
        tr_gen, validation_data=vl_gen,
        epochs=EPOCHS, callbacks=[LR_CB], verbose=1,
    )
    train_time = time.time() - t0

    # Save weights
    wpath = f'{WEIGHTS}/fold{k}.weights.h5'
    model.save_weights(wpath)

    # Predict on validation (full — not HQ-only — so we can compute both metrics)
    yp = model.predict(vl_gen, verbose=0).astype(np.float32)
    yt = vl_df[TARGETS].values.astype(np.float32)[:len(yp)]
    wt = vl_df['total_votes_sum'].values.astype(np.float32)[:len(yp)]
    max_seg_votes = vl_df['max_segment_votes'].values.astype(np.float32)[:len(yp)]
    eeg_ids_val = vl_df['eeg_id'].values[:len(yp)]
    # spec_id column name is 'spec_id' per cell 5 of this notebook
    spec_ids_val = vl_df['spec_id'].values[:len(yp)] if 'spec_id' in vl_df.columns else eeg_ids_val.copy()

    kld_all = kld_np(yp, yt)
    hq_mask = max_seg_votes >= 10
    kld_hq  = kld_np(yp[hq_mask], yt[hq_mask]) if hq_mask.any() else float('nan')

    print(f'  Fold {k}: KLD(all)={kld_all:.4f}  KLD(HQ, n={hq_mask.sum()})={kld_hq:.4f}  '
          f'time={train_time:.0f}s')

    # Recover equivalent logits from softmax outputs. log(p) differs from true
    # pre-softmax logits by a constant per sample, which is normalized away by
    # any downstream softmax operation — so this is equivalent for KLD, argmax,
    # AUROC, etc. Keeps the schema consistent with VIPEEGNet and TCS-Net.
    yp_raw_logits = np.log(np.clip(yp, 1e-8, 1.0)).astype(np.float32)

    # Save per-fold OOF in the shared format
    np.savez(
        f'{OUT_DIR}/fold{k}_oof.npz',
        yt=yt, yp=yp, wt=wt,
        yp_raw_logits=yp_raw_logits,
        eeg_ids=eeg_ids_val,
        spectrogram_ids=spec_ids_val,
        max_segment_votes=max_seg_votes,
        hq_mask=hq_mask.astype(bool),
        temperature=np.array([1.0], dtype=np.float32),
    )

    summary['per_fold'].append({
        'fold':            int(k),
        'n_valid':         int(len(vl_df)),
        'n_hq':            int(hq_mask.sum()),
        'kld_all':         float(kld_all),
        'kld_hq_native':   float(kld_hq),  # no TTA, no temp → native == raw
        'kld_hq_raw':      float(kld_hq),
        'kld_hq':          float(kld_hq),  # backward-compat alias
        'temperature':     1.0,
        'train_seconds':   float(train_time),
        'history_loss':    [float(x) for x in hist.history.get('loss', [])],
        'history_val_loss':[float(x) for x in hist.history.get('val_loss', [])],
    })

    del model, tr_gen, vl_gen, yp, yt
    tf.keras.backend.clear_session()
    gc.collect()

total_time = time.time() - t_start_all
summary['total_seconds'] = float(total_time)
print(f'\nAll folds complete. Wall time: {total_time/60:.1f} min')



########################################
### Fold 0
########################################
Train 13,212 eeg_ids | Valid 3,877 eeg_ids

Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/4


I0000 00:00:1776876867.569869  123761 service.cc:152] XLA service 0x7c810c015a30 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1776876867.569914  123761 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4090, Compute Capability 8.9
I0000 00:00:1776876871.789269  123761 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1776876904.077564  123761 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


413/413 ━━━━━━━━━━━━━━━━━━━━ 201s 349ms/step - loss: 0.8377 - val_loss: 4.3105 - learning_rate: 0.0010

Epoch 2: LearningRateScheduler setting learning rate to 0.001.
Epoch 2/4
413/413 ━━━━━━━━━━━━━━━━━━━━ 86s 208ms/step - loss: 0.5483 - val_loss: 3.0527 - learning_rate: 0.0010

Epoch 3: LearningRateScheduler setting learning rate to 0.0001.
Epoch 3/4
413/413 ━━━━━━━━━━━━━━━━━━━━ 82s 198ms/step - loss: 0.4198 - val_loss: 1.7781 - learning_rate: 1.0000e-04

Epoch 4: LearningRateScheduler setting learning rate to 1.0000000000000003e-05.
Epoch 4/4
413/413 ━━━━━━━━━━━━━━━━━━━━ 81s 196ms/step - loss: 0.3527 - val_loss: 1.0159 - learning_rate: 1.0000e-05
  Fold 0: KLD(all)=1.0159  KLD(HQ, n=1296)=0.8857  time=451s

########################################
### Fold 1
########################################
Train 13,989 eeg_ids | Valid 3,100 eeg_ids

Epoch 1: LearningRateScheduler setting learning rate to 0.001.
Epoch 1/4
438/438 ━━━━━━━━━━━━━━━━━━━━ 163s 274ms/step - loss: 0.8170 - val_loss:

## 8. Aggregate OOF and save the shared summary


In [ ]:
# OOF concat — shared-schema aggregation
all_yt, all_yp, all_w, all_hq, all_ids, all_sids, all_logits = [], [], [], [], [], [], []
for k in range(N_FOLDS):
    d = np.load(f'{OUT_DIR}/fold{k}_oof.npz', allow_pickle=True)
    all_yt.append(d['yt']); all_yp.append(d['yp']); all_w.append(d['wt'])
    all_hq.append(d['hq_mask']); all_ids.append(d['eeg_ids'])
    all_sids.append(d['spectrogram_ids'])
    all_logits.append(d['yp_raw_logits'])
all_yt     = np.concatenate(all_yt)
all_yp     = np.concatenate(all_yp)
all_w      = np.concatenate(all_w)
all_hq     = np.concatenate(all_hq)
all_ids    = np.concatenate(all_ids)
all_sids   = np.concatenate(all_sids)
all_logits = np.concatenate(all_logits)

oof_kld_all = kld_np(all_yp, all_yt)
oof_kld_hq  = kld_np(all_yp[all_hq], all_yt[all_hq])

# EfficientNetB0 has no TTA or temperature, so native == raw
summary['oof_kld_all']         = float(oof_kld_all)
summary['oof_kld_hq']          = float(oof_kld_hq)
summary['oof_kld_hq_native']   = float(oof_kld_hq)
summary['oof_kld_hq_raw']      = float(oof_kld_hq)
summary['n_oof_total']         = int(len(all_yt))
summary['n_oof_hq']            = int(all_hq.sum())

print(f'OOF KLD (all eeg_ids, n={len(all_yt):,}):     {oof_kld_all:.4f}')
print(f'OOF KLD (HQ eeg_ids,  n={all_hq.sum():,}):     {oof_kld_hq:.4f}')
print(f'Per-fold KLD(HQ): {[f"{f[\"kld_hq\"]:.4f}" for f in summary["per_fold"]]}')
print(f'  mean: {np.mean([f["kld_hq"] for f in summary["per_fold"]]):.4f}')
print(f'\nDeotte reports CV 0.59 KLD on this recipe (LB 0.43-0.44).')

# Save summary
with open(f'{OUT_DIR}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved {OUT_DIR}/summary.json')
print(f'Saved {OUT_DIR}/fold{{0..{N_FOLDS-1}}}_oof.npz')
print(f'Saved {OUT_DIR}/weights/fold{{0..{N_FOLDS-1}}}.weights.h5')


## 9. Per-class diagnostic (AUROC + confusion)

Quick sanity check. The comparison notebook will produce the full cross-model figures later.


In [ ]:
# HQ subset only — that's the comparable-across-models set
# Guard against all_yt/all_yp being lists if cell 21 wasn't re-run
if isinstance(all_yt, list):
    all_yt = np.concatenate(all_yt)
if isinstance(all_yp, list):
    all_yp = np.concatenate(all_yp)
if isinstance(all_hq, list):
    all_hq = np.concatenate(all_hq)

import seaborn as sns
from sklearn.metrics import confusion_matrix

# HQ subset only — that's the comparable-across-models set
yt = all_yt[all_hq]
yp = all_yp[all_hq]
yti = yt.argmax(1); ypi = yp.argmax(1)
ytb = (yti[:, None] == np.arange(6)).astype(int)

print(f'{"Class":>8} | {"AUROC":>6}')
print('-' * 25)
for i, nm in enumerate(CLASS_NAMES):
    try:
        auc = roc_auc_score(ytb[:, i], yp[:, i])
        print(f'{nm:>8} | {auc:.3f}')
    except Exception as e:
        print(f'{nm:>8} | n/a ({e})')

cm = confusion_matrix(yti, ypi, labels=range(6))
cm_r = cm.astype('float') / cm.sum(1, keepdims=True) * 100
cm_p = cm.astype('float') / cm.sum(0, keepdims=True) * 100

fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 6))
sns.heatmap(cm_r, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=a1)
a1.set_title('EfficientNetB0 Recall (%)')
sns.heatmap(cm_p, annot=True, fmt='.1f', cmap='Oranges',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=a2)
a2.set_title('EfficientNetB0 Precision (%)')
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/confusion.png', dpi=120)
plt.show()
print(f'\nSaved {OUT_DIR}/confusion.png')

## Done

Outputs saved to `/workspace/fair_comparison/effnetb0/`:
- `summary.json` — per-fold and OOF KLD, config
- `fold{0..4}_oof.npz` — yt, yp, wt, eeg_ids, hq_mask
- `weights/fold{0..4}.weights.h5`
- `confusion.png`

When TCS-Net and VIPEEGNet finish, the comparison notebook can load all three from their
respective `fair_comparison/<model>/` directories and produce the cross-model plots.
